In [ ]:
!pip install -q fastai timm transformers scipy tacoreader rasterio


In [ ]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

!PYTHONPATH={SRC_DIR} python -m pytest tests -q


In [ ]:
from pathlib import Path

import pandas as pd

from cloudband.colab.session import start
from cloudband.datasets import cloudsen12 as dataset
from cloudband.datasets.cloudsen12 import iter_samples
from cloudband.eval.metrics import balanced_overall_accuracy
from cloudband.models.ocm import OcmEnsemble, run_ocm_ensemble_phase2
from cloudband.models.swin_upernet import run_swin_upernet_phase2
from cloudband.pipelines.cloudsen12 import per_scene_metric, score_split
from cloudband.provenance.manifest import file_digest
from cloudband.train.comparison import compare_across_seeds
from cloudband.train.data import build_dataloaders
from cloudband.train.predictor import build_predictor
from cloudband.train.protocol import ocm_shared_protocol, swin_shared_protocol


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

RESULTS_DIR = Path("/content/drive/MyDrive/cloudband/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

session = start(PROJECT_DIR, require_accelerator=True)
print(f"results: {RESULTS_DIR}")


In [ ]:
def load_split(remote_source, split):
    table = dataset.load_remote(remote_source)
    differences = dataset.verify_split_sizes(table)
    if differences:
        print("split sizes differ from documented ones:", differences)
    return dataset.select(table, split=split)


train_l1c = load_split(dataset.REMOTE_L1C, split="train")
train_l2a = load_split(dataset.REMOTE_L2A, split="train")
train_table = pd.concat([train_l1c, train_l2a], ignore_index=True)

valid_l1c = load_split(dataset.REMOTE_L1C, split="validation")
valid_l2a = load_split(dataset.REMOTE_L2A, split="validation")
valid_table = pd.concat([valid_l1c, valid_l2a], ignore_index=True)

test_table = load_split(dataset.REMOTE_L1C, split="test")

print("train patches:", len(train_table))
print("validation patches:", len(valid_table))
print("test patches:", len(test_table))


In [ ]:
MICRO_BATCH_SIZE = 8
NUM_WORKERS = 2

train_valid_dls = build_dataloaders(
    train_table=train_table,
    valid_table=valid_table,
    micro_batch_size=MICRO_BATCH_SIZE,
    num_workers=NUM_WORKERS,
)


In [ ]:
def save_run_to_drive(run, checkpoint_source_dir):
    manifest_path = run.manifest.write(RESULTS_DIR / f"{run.manifest.result_name}.json")

    checkpoint_source = Path(checkpoint_source_dir) / f"{run.fit_result.checkpoint_name}.pth"
    checkpoint_target = CHECKPOINT_DIR / checkpoint_source.name
    checkpoint_target.write_bytes(checkpoint_source.read_bytes())

    print(run.manifest.result_name)
    print("  manifest:", manifest_path)
    print("  checkpoint:", checkpoint_target)
    print("  checkpoint sha256:", file_digest(checkpoint_target))


In [ ]:
ocm_protocol = ocm_shared_protocol(learning_rate=1e-4)  # placeholder, the search overwrites it
ocm_runs = run_ocm_ensemble_phase2(train_valid_dls, ocm_protocol, seeds=ocm_protocol.seeds)

for backbone_name, seed_runs in ocm_runs.items():
    for seed, run in seed_runs.items():
        print(backbone_name, "seed", seed)
        print("  winning learning rate:", run.winning_protocol.learning_rate)
        print("  best validation loss:", run.fit_result.best_val_loss)
        save_run_to_drive(run, Path(PROJECT_DIR) / "models")


In [ ]:
swin_protocol = swin_shared_protocol(learning_rate=1e-4)  # placeholder
swin_runs = run_swin_upernet_phase2(train_valid_dls, swin_protocol, seeds=swin_protocol.seeds)

for run in swin_runs:
    print("seed", run.fit_result.seed)
    print("  winning learning rate:", run.winning_protocol.learning_rate)
    print("  best validation loss:", run.fit_result.best_val_loss)
    save_run_to_drive(run, Path(PROJECT_DIR) / "models")


In [ ]:
ocm_boa_per_seed = []
for seed in ocm_protocol.seeds:
    ocm_ensemble_model = OcmEnsemble(
        tuple(
            ocm_runs[backbone_name][seed].fit_result.learner.model
            for backbone_name in ocm_runs
        )
    )
    ocm_predictor = build_predictor(ocm_ensemble_model)
    ocm_per_scene = score_split(iter_samples(test_table), ocm_predictor)
    boa = per_scene_metric(ocm_per_scene, balanced_overall_accuracy)
    boa.to_csv(RESULTS_DIR / f"ocm_shared_per_scene_boa_seed{seed}.csv")
    ocm_boa_per_seed.append(boa)

swin_boa_per_seed = []
for run in swin_runs:
    swin_predictor = build_predictor(run.fit_result.learner.model)
    swin_per_scene = score_split(iter_samples(test_table), swin_predictor)
    boa = per_scene_metric(swin_per_scene, balanced_overall_accuracy)
    boa.to_csv(RESULTS_DIR / f"swin_shared_per_scene_boa_seed{run.fit_result.seed}.csv")
    swin_boa_per_seed.append(boa)

print("ocm mean BOA per seed:")
for boa in ocm_boa_per_seed:
    print(boa.mean().to_dict())
print("swin mean BOA per seed:")
for boa in swin_boa_per_seed:
    print(boa.mean().to_dict())


In [ ]:
results = compare_across_seeds(tuple(ocm_boa_per_seed), tuple(swin_boa_per_seed))

for result in results:
    print(result.experiment)
    print("  pairable scenes:", result.n_pairs)
    print("  p-value:", result.p_value)
    print("  median difference, ocm minus swin:", result.median_difference)
